# 08 · Type Hints & Readable Code — **Depth**

> **Pull this notebook when:** you want code a reviewer (or future-you) reads without friction —
> Week 10 explicitly asks for type hints and clean architecture. This is the readability bar.

Depth here is what annotations *are* (and aren't), the type vocabulary, and the judgment of clean code.
Predict first.

---

## 8.1 — Predict: are type hints enforced?

```python
def add(a: int, b: int) -> int:
    return a + b

print(add(2, 3))          # ?
print(add("x", "y"))      # ? passing strings to an int-annotated function
print(add.__annotations__)  # ? where do the hints live
```

In [ ]:
def add(a: int, b: int) -> int:
    return a + b

print(add(2, 3))
print(add("x", "y"))
print(add.__annotations__)

<details>
<summary>▶ Reveal</summary>

`5`, then `"xy"`, then `{'a': <class 'int'>, 'b': <class 'int'>, 'return': <class 'int'>}`.

Type hints are **not enforced at runtime**. `add("x", "y")` runs fine and concatenates the strings —
Python ignores the annotations during execution. They're stored in `__annotations__` as metadata,
*for tools* (type checkers like mypy/pyright, IDEs, documentation) — not the interpreter. This is
"gradual typing": hints are documentation and a static-analysis aid, not runtime guarantees. So they
don't slow your code or change behavior; their value is that a checker (and a reader) catches the
`add("x", "y")` mistake *before* you run, and a reader understands the contract at a glance.

</details>

## 8.2 — Predict: structural typing with `Protocol`

`Protocol` defines a *shape* — any object with the right methods satisfies it, **without inheriting**
it. With `@runtime_checkable`, `isinstance` checks for the methods' presence.

```python
from typing import Protocol, runtime_checkable

@runtime_checkable
class Speaker(Protocol):
    def speak(self) -> str: ...

class Dog:                     # does NOT inherit Speaker
    def speak(self) -> str:
        return "woof"

class Rock:
    pass

print(isinstance(Dog(), Speaker))    # ?
print(isinstance(Rock(), Speaker))   # ?
```

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Speaker(Protocol):
    def speak(self) -> str: ...

class Dog:
    def speak(self) -> str:
        return "woof"

class Rock:
    pass

print(isinstance(Dog(), Speaker))
print(isinstance(Rock(), Speaker))

<details>
<summary>▶ Reveal</summary>

`True`, then `False`.

`Dog` satisfies `Speaker` **without inheriting it** — it just has a `speak` method, and that's all a
`Protocol` requires. This is **structural typing** ("if it has the methods, it counts"), the typed
expression of duck typing you met in notebook 05. `Rock` has no `speak`, so it fails. `Protocol` lets
you annotate "I accept anything that can `speak`" without forcing every caller to inherit a base class —
much more flexible than nominal typing. You'd use it to type "any sampler with a `.sample()` method" or
"any component with a `.forward()`" without a rigid inheritance tree.

</details>

## 8.3 — Build + annotate: three ways to express "maybe a value"

A lookup may or may not find a result. Annotate and implement three styles (answer covers when each):
`Optional` return, a default-`None` parameter, and a union return.

In [ ]:
from typing import Optional

def find_optional(items, target) -> Optional[int]:
    # return the index of target, or None if absent
    # YOUR CODE HERE
    pass

def get_or_default(d: dict, key: str, default: Optional[str] = None) -> Optional[str]:
    # return d[key] if present, else default
    # YOUR CODE HERE
    pass

def parse_int(s: str) -> "int | str":
    # return int(s), or the string "invalid" if it can't parse
    # YOUR CODE HERE
    pass

# Tests
assert find_optional([10, 20, 30], 20) == 1
assert find_optional([10, 20], 99) is None
assert get_or_default({"a": "x"}, "a") == "x"
assert get_or_default({"a": "x"}, "b") is None
assert get_or_default({"a": "x"}, "b", "fallback") == "fallback"
assert parse_int("42") == 42
assert parse_int("nope") == "invalid"
print("8.3 passed")

<details>
<summary>▶ When each</summary>

- **`Optional[int]`** (same as `int | None`) — the standard way to say "an int, or nothing." Use for
  lookups/searches that may fail to find something. The caller knows to check for `None`.
- **default-`None` parameter** — for *optional inputs*: the caller may omit it. Pair with the
  `None`-sentinel pattern (never a mutable default — notebook 01).
- **union return `int | str`** — when a function genuinely returns *different types*. Often a smell
  (forcing the caller to type-check the result); usually better to raise an exception or return
  `Optional`. Included so you recognize when a union is honest vs when it's hiding a messy contract.

Modern syntax: `int | None` and `int | str` (Python 3.10+) read better than `Optional`/`Union`. Prefer
them; know `Optional`/`Union` because older code uses them.

</details>

## ★ Milestone 08 — Refactor to industry-standard readability

The capstone of the whole depth set. Below is a working but **badly written** function: cryptic names,
a positional boolean flag, no type hints, a bare `except`, and a mutable default argument. Its behavior
is locked by the test suite — your job is to **refactor it to clean, industry-standard code without
changing what it does.**

The messy original (study it, then rewrite in the cell):

```python
def p(d, k, u=[], f=False):
    try:
        v = d[k]
    except:
        return None
    if f:
        v = v.upper()
    u.append(v)
    return u
```

What it does: looks up key `k` in dict `d`; returns `None` if missing; optionally uppercases the value;
appends it to a list and returns the list.

Rewrite it as `collect_value` with: **clear names**, **full type hints**, the flag as a **keyword-only**
argument, a **specific exception** (not bare), the **mutable-default fixed** (None sentinel), and a
**docstring**. Behavior must match the locked tests exactly.

(rewrite below — match the behavior the tests pin down)

In [ ]:
from typing import Optional

def collect_value(
    # YOUR CODE HERE — clean signature: source dict, key, an optional accumulator list,
    # and a keyword-only uppercase flag; full annotations
):
    """YOUR DOCSTRING HERE"""
    # YOUR CODE HERE — look up key (specific exception on miss -> return None),
    # optional uppercase, append to a fresh-or-given list, return the list
    pass

# Tests
# behavior is locked — the refactor must match the original exactly

# basic lookup + append to a fresh list each call (mutable-default bug fixed)
assert collect_value({"a": "x"}, "a") == ["x"]
assert collect_value({"a": "x"}, "a") == ["x"]      # fresh list, NOT ["x", "x"]

# missing key returns None
assert collect_value({"a": "x"}, "missing") is None

# uppercase flag, now keyword-only
assert collect_value({"a": "hi"}, "a", uppercase=True) == ["HI"]

# accumulator can be supplied and is appended to
acc = ["existing"]
assert collect_value({"a": "x"}, "a", acc) == ["existing", "x"]
assert acc == ["existing", "x"]

# the flag truly is keyword-only now
raised = False
try:
    eval('collect_value({"a": "x"}, "a", None, True)')   # positional flag -> TypeError
except TypeError:
    raised = True
assert raised

# it's annotated for readers
anns = collect_value.__annotations__
assert "source" in anns and "return" in anns
assert collect_value.__doc__ is not None and len(collect_value.__doc__) > 20
print("Milestone 08 passed")

<details>
<summary>▶ What changed and why</summary>

Every change is a readability/correctness standard from across the depth set:

- **Names**: `p`→`collect_value`, `d`→`source`, `k`→`key`, `u`→`accumulator`, `f`→`uppercase`, `v`→
  `value`. A reader knows the contract from the signature alone.
- **Type hints**: `dict[str, str]`, `Optional[list[str]]`, `bool`, return `Optional[list[str]]` — the
  whole contract is documented and checkable (8.1).
- **Keyword-only flag**: the bare `*` forces `uppercase=True` at call sites, so no mystery positional
  booleans (notebook 02.7).
- **Specific exception**: `except KeyError` instead of bare `except`, which would have swallowed
  *everything* including bugs (notebook 06.2).
- **Mutable default fixed**: `accumulator=None` + create-inside, so each call gets a fresh list instead
  of the shared one that caused the original to accumulate across calls (notebook 01.3).
- **Docstring**: states what it returns, the `None` case, and the side effect.

This is the Week 10 "clean architecture" standard in one function. The behavior is byte-identical to the
mess; only the readability and safety changed — which is exactly what a real refactor is.

</details>